# TM-align scores: reconstructed vs original PDBs

Pairs each **`pdbs/reconstructed/*_reconstructed_pair_structure.pdb`** with **`pdbs/original/*_pair_block_47_structure.pdb`** (same protein id) and runs **TMalign**.

Defaults assume this notebook lives in **`AlphaFold_Autoencoder/original_SAE/`** next to sibling **`pdbs/`**. Override with env **`PRED_PDB_DIR`**, **`REF_PDB_DIR`**, **`TMALIGN_BIN`**, **`TM_SCORES_OUTPUT`**.

If **TMalign** is not on `PATH`, the next cell can **compile** it into **`AlphaFold_Autoencoder/bin/`** (needs `g++`). It uses **`TMalign.cpp` in `original_SAE/`** (same folder as this notebook) if you put the file there—no download—or else tries to download once. You can still set **`TMALIGN_BIN`** to an existing binary.

In [ ]:
from __future__ import annotations

import csv
import glob
import json
import os
import re
import subprocess
from pathlib import Path


def _find_script_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "compute_tm_scores_token_sae.py").is_file():
            return p
        p = p.parent
    return Path.cwd().resolve()


SCRIPT_DIR = _find_script_dir()
AUTOENCODER_ROOT = Path(os.environ.get("AUTOENCODER_ROOT", str(SCRIPT_DIR.parent))).resolve()
print("AUTOENCODER_ROOT:", AUTOENCODER_ROOT)
print("SCRIPT_DIR:", SCRIPT_DIR)

In [ ]:
import sys

# TM-align: use existing PATH/bin/TMALIGN_BIN, or download + g++ into AUTOENCODER_ROOT/bin/
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
from compute_tm_scores_token_sae import ensure_tmalign_or_build

# --- Paths (override with env) ---
PRED_DIR = Path(
    os.environ.get("PRED_PDB_DIR", str(AUTOENCODER_ROOT / "pdbs" / "reconstructed"))
).resolve()
REF_DIR = Path(os.environ.get("REF_PDB_DIR", str(AUTOENCODER_ROOT / "pdbs" / "original"))).resolve()
OUTPUT_JSON = Path(
    os.environ.get("TM_SCORES_OUTPUT", str(AUTOENCODER_ROOT / "tm_scores" / "tm_scores.json"))
).resolve()

_raw_tm = os.environ.get("TMALIGN_BIN", "TMalign")
_allow_build = os.environ.get("TMALIGN_NO_AUTO_BUILD", "").strip() not in ("1", "true", "yes")
TMALIGN_BIN = ensure_tmalign_or_build(
    _raw_tm,
    search_roots=(SCRIPT_DIR, AUTOENCODER_ROOT),
    install_under=AUTOENCODER_ROOT,
    allow_build=_allow_build,
)

TMSCORE_RE = re.compile(r"TM-score\s*=\s*([\d.]+)")

print("PRED_DIR:", PRED_DIR)
print("REF_DIR:", REF_DIR)
print("OUTPUT_JSON:", OUTPUT_JSON)
print("TMALIGN_BIN:", TMALIGN_BIN)

In [ ]:
def run_tmalign(pred_pdb: str, ref_pdb: str, tmalign_bin: str) -> float | None:
    """TMalign(ref, pred) with -ter 0; return first TM-score in stdout (normalized by ref)."""
    try:
        result = subprocess.run(
            [tmalign_bin, ref_pdb, pred_pdb, "-ter", "0"],
            capture_output=True,
            text=True,
            timeout=120,
        )
        if result.returncode != 0:
            return None
        for line in result.stdout.splitlines():
            m = TMSCORE_RE.search(line)
            if m:
                return float(m.group(1))
    except Exception as e:
        print(f"Error running TMalign on {pred_pdb}: {e}")
    return None


if not PRED_DIR.is_dir():
    raise FileNotFoundError(f"Pred dir not found: {PRED_DIR}")
if not REF_DIR.is_dir():
    raise FileNotFoundError(f"Ref dir not found: {REF_DIR}")

pred_files = sorted(glob.glob(str(PRED_DIR / "*_reconstructed_pair_structure.pdb")))
scores: dict[str, float] = {}

print(f"Calculating TM-scores for {len(pred_files)} structures...")

for pred_path in pred_files:
    filename = os.path.basename(pred_path)
    protein_id = filename.replace("_reconstructed_pair_structure.pdb", "")
    ref_path = REF_DIR / f"{protein_id}_pair_block_47_structure.pdb"
    if ref_path.is_file():
        score = run_tmalign(pred_path, str(ref_path), TMALIGN_BIN)
        if score is not None:
            scores[protein_id] = score
            print(f"  {protein_id}: {score:.4f}")
        else:
            print(f"  {protein_id}: TMalign failed (nonzero exit or no parse)")
    else:
        print(f"  Warning: missing reference for {protein_id} at {ref_path}")

if not scores:
    raise RuntimeError("No valid TM-scores — check TMalign, PDB names, and ref/pred dirs.")

valid_scores = list(scores.values())
avg_score = sum(valid_scores) / len(valid_scores)
print(f"\nTotal aligned: {len(valid_scores)}")
print(f"Average TM-score: {avg_score:.4f}")

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
data = {
    "average_tm_score": avg_score,
    "total_evaluated": len(scores),
    "pred_dir": str(PRED_DIR),
    "ref_dir": str(REF_DIR),
    "individual_scores": scores,
}
with open(OUTPUT_JSON, "w") as f:
    json.dump(data, f, indent=2)
print(f"JSON saved to {OUTPUT_JSON}")

csv_path = OUTPUT_JSON.with_suffix(".csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["protein_id", "tm_score"])
    for pid, s in sorted(scores.items()):
        w.writerow([pid, f"{s:.6f}"])
print(f"CSV saved to {csv_path}")